<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/04-spark-sql/02-catalog-api.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 4.2 — The Catalog API

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("4.2-catalog-api")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True).schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)
orders.createOrReplaceTempView("orders")

logs = spark.read.text(f"{DATA_DIR}/sample_logs.txt")
logs.createOrReplaceTempView("raw_logs")

## The read side — all metadata, no jobs

In [ ]:
print("current db:", spark.catalog.currentDatabase())
print("databases: ", [d.name for d in spark.catalog.listDatabases()])

for t in spark.catalog.listTables():
    print(f"{t.name:<10} type={t.tableType:<10} isTemporary={t.isTemporary}")

In [ ]:
for c in spark.catalog.listColumns("orders"):
    print(f"{c.name:<12} {c.dataType:<8} nullable={c.nullable}")

print()
print("orders exists:", spark.catalog.tableExists("orders"))
print("sales exists: ", spark.catalog.tableExists("sales"))

In [ ]:
# catalog name -> DataFrame (inverse of createOrReplaceTempView)
df_again = spark.table("orders")
df_again.count()

# SQL twins return DataFrames instead of Python lists:
spark.sql("SHOW TABLES").show()
spark.sql("DESCRIBE orders").show(4)

## The write side — dropping and caching by name

In [ ]:
spark.catalog.cacheTable("orders")
spark.table("orders").count()                      # materializes (lesson 2.4)
print("cached:", spark.catalog.isCached("orders"), "-> see Storage tab")

spark.catalog.uncacheTable("orders")
print("cached:", spark.catalog.isCached("orders"))

print("dropped raw_logs:", spark.catalog.dropTempView("raw_logs"))
print("dropped again:   ", spark.catalog.dropTempView("raw_logs"))   # False, no error

## The payoff: a generic profiler over WHATEVER exists

In [ ]:
def profile_all_tables(spark):
    """Null-count report for every table/view in the session. No names hard-coded."""
    for t in spark.catalog.listTables():
        df = spark.table(t.name)
        cols = [c.name for c in spark.catalog.listColumns(t.name)]
        nulls = df.select([
            F.count(F.when(col(c).isNull(), 1)).alias(c) for c in cols
        ]).first().asDict()
        flagged = {k: v for k, v in nulls.items() if v > 0}
        print(f"{t.name}: rows={df.count()}, null_columns={flagged or 'none'}")

# register a second view so the loop has something to loop over
spark.table("orders").filter(col("category") == "books").createOrReplaceTempView("book_orders")

profile_all_tables(spark)

## Exercises

1. Write `assert_schema(spark, name, expected: dict)` — raise if a column is missing or has the wrong dataType. Test it against `orders` with one deliberate mismatch.
2. Extend the profiler to also report each table's numeric columns' min/max (build the agg list from `listColumns` dataTypes).
3. PII scanner: flag any table with columns matching `customer|email|phone` in the name. Run it — which of ours trips?
4. How many built-in functions does `listFunctions()` report? Find three whose names you don't recognize and look them up — free F-namespace vocabulary (lesson 3.2 habit).

In [ ]:
# your exercise code here
